In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
from torchsummary import summary
import pickle

In [2]:
with open('../dataset/train.pkl', 'rb') as f:
    X_train, y_train = pickle.load(f)

print("Train data:")
print("X_train shape: ", X_train.shape) # expect (12800, 96, 96) 15999
print("y_train shape: ", y_train.shape) # (12800, )

X_train = X_train[..., np.newaxis]
X_train = np.transpose(X_train, (0, 3, 1, 2))

print("X_train shape: ", X_train.shape)

Train data:
X_train shape:  (11200, 96, 96)
y_train shape:  (11200,)
X_train shape:  (11200, 1, 96, 96)


In [3]:
class InvertedResidual(nn.Module):
    def __init__(self, in_c, out_c, stride, expand_ratio):
        super().__init__()
        hidden_dim = in_c * expand_ratio
        self.use_res = stride == 1 and in_c == out_c

        layers = []
        
        if expand_ratio != 1:
            layers.extend([
                nn.Conv2d(in_c, hidden_dim, 1, bias=False),
                nn.BatchNorm2d(hidden_dim),
                nn.ReLU6(inplace=True)
            ])
        
        layers.extend([
            nn.Conv2d(hidden_dim, hidden_dim, 3, stride, 1, groups=hidden_dim, bias=False),
            nn.BatchNorm2d(hidden_dim),
            nn.ReLU6(inplace=True),
            nn.Conv2d(hidden_dim, out_c, 1, bias=False),
            nn.BatchNorm2d(out_c)
        ])

        self.conv = nn.Sequential(*layers)

    def forward(self, x):
        if self.use_res:
            return x + self.conv(x)
        return self.conv(x)

In [4]:
class LightGestureNet(nn.Module):
    def __init__(self, num_classes=8):
        super().__init__()

        self.first = nn.Sequential(
            nn.Conv2d(1, 16, 3, 2, 1, bias=False),
            nn.BatchNorm2d(16),
            nn.ReLU6(inplace=True)
        )

        self.layers = nn.Sequential(
            InvertedResidual(16, 24, 2, 6),
            InvertedResidual(24, 24, 1, 6),
            InvertedResidual(24, 32, 2, 6),
            InvertedResidual(32, 32, 1, 6),
        )

        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Linear(32, num_classes)
        )

    def forward(self, x):
        x = self.first(x)
        x = self.layers(x)
        x = self.classifier(x)
        return x

In [5]:
model = LightGestureNet()

print(model)

sample_input = torch.randn(1, 1, 96, 96)
print("Sample input shape: ", sample_input.shape)
output = model(sample_input)
print("Output shape: ", output.shape)


# Torch summary
print("\nSummary: ")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = LightGestureNet().to(device)

summary(model, input_size=(1, 96, 96), device=str(device))

LightGestureNet(
  (first): Sequential(
    (0): Conv2d(1, 16, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
    (1): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU6(inplace=True)
  )
  (layers): Sequential(
    (0): InvertedResidual(
      (conv): Sequential(
        (0): Conv2d(16, 96, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (1): BatchNorm2d(96, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (2): ReLU6(inplace=True)
        (3): Conv2d(96, 96, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), groups=96, bias=False)
        (4): BatchNorm2d(96, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (5): ReLU6(inplace=True)
        (6): Conv2d(96, 24, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (7): BatchNorm2d(24, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      )
    )
    (1): InvertedResidual(
      (conv): Sequential(
        (0)